# ARC v0.20d — HNSW Severity Sensitivity Audit

**Purpose.** Test whether the v0.20c HNSW reversal persists under milder search-effort contrasts without altering the frozen primary HNSW result.

Primary v0.20c remains immutable:

- HNSW `M=32`, `efConstruction=200`
- primary contrast: `efSearch 8 -> 256`
- FEVER + E5-small-v2
- 44 frozen feedback policies
- 4 feedback updates
- top-100 retrieval
- nDCG@10
- epsilon = 0.002
- query-level statistical unit

This notebook is a **post-primary exploratory severity sensitivity audit**. It evaluates:

- `efSearch 32 -> 256`
- `efSearch 64 -> 256`

No HNSW index is rebuilt. The existing v0.20c index and frozen FEVER/E5 lineage are reused.

### Scientific interpretation

This audit does **not** replace the primary `8 -> 256` result and must not be described as preregistered confirmation. Its purpose is narrower: determine whether the qualitative sign pattern `H1 > 0, H2 < 0, H3abs < 0` persists when the one-shot fidelity gap is reduced.

The contrast list is written to a sealed audit protocol file before trajectory outcomes are computed.


In [ ]:
%pip install -q faiss-cpu==1.12.0 pyarrow psutil

from pathlib import Path
import os, json, hashlib, time, gc, sys
import numpy as np
import pandas as pd
import faiss
import psutil

print("Python :", sys.version.split()[0])
print("FAISS  :", faiss.__version__)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))
assert faiss.__version__ == "1.12.0"
print("ENVIRONMENT CHECK: PASS")


## 1. Mount Drive and resolve v0.20c + frozen source lineage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints")
ARC_ROOT = DRIVE_ROOT / "arc-v0"
V020C = ARC_ROOT / "hnsw-mechanism-replication-v020c"
INDEX_PATH = V020C / "fever-e5-small-v2-hnswflat-m32-efc200.faiss"
DOC_IDS_PATH = V020C / "fever-e5-doc-ids.txt"
PRIMARY_FREEZE = V020C / "V020C_FROZEN_CONTRAST.json"
PRIMARY_REPORT = V020C / "v020c_final_report.json"
ONE_SHOT_LADDER = V020C / "v020c_fit_one_shot_ladder.csv"
BUILD_RECORD = V020C / "v020c_hnsw_build_record.json"

V018_RUN = ARC_ROOT / "cross-encoder-fever-replication-v018" / "20260819-015645"
V018_SHARDS = V018_RUN / "corpus_shards"
V018_QUERY_EMB = V018_RUN / "dev_query_embeddings.float32.npy"
V018_QUERY_IDS = V018_RUN / "dev_query_ids.txt"
V018_PROTOCOL = V018_RUN / "v018_cross_encoder_protocol.json"
V018_MANIFEST = V018_RUN / "corpus_encoding_manifest.json"
V013_SPLIT_CANDIDATES = [
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-151852" / "v013_boundary_query_split.csv",
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-140640" / "v013_boundary_query_split.csv",
]
V013_SPLIT = next((p for p in V013_SPLIT_CANDIDATES if p.exists()), None)
FEVER_QRELS_DEV = DRIVE_ROOT / "raw-datasets" / "fever" / "qrels" / "dev.tsv"
OUT = ARC_ROOT / "hnsw-severity-sensitivity-v020d"
OUT.mkdir(parents=True, exist_ok=True)

required = [INDEX_PATH, DOC_IDS_PATH, PRIMARY_FREEZE, PRIMARY_REPORT, ONE_SHOT_LADDER,
            BUILD_RECORD, V018_QUERY_EMB, V018_QUERY_IDS, V018_PROTOCOL, V018_MANIFEST,
            V013_SPLIT, FEVER_QRELS_DEV]
missing = [str(p) for p in required if p is None or not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing))
print("V020C :", V020C)
print("OUT   :", OUT)
print("DRIVE PATH CHECK: PASS")


## 2. Provenance audit and freeze the secondary contrast list

In [ ]:
def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

primary_freeze = json.loads(PRIMARY_FREEZE.read_text())
primary_report = json.loads(PRIMARY_REPORT.read_text())
build_record = json.loads(BUILD_RECORD.read_text())
v018_protocol = json.loads(V018_PROTOCOL.read_text())
assert primary_freeze["low_efSearch"] == 8
assert primary_freeze["high_efSearch"] == 256
assert primary_freeze["selected_before_feedback_outcomes"] is True
assert primary_report["mechanism_gate"] == "HNSW_REVERSAL"
assert build_record["index_sha256"] == sha256_file(INDEX_PATH)
assert v018_protocol["encoder"] == "intfloat/e5-small-v2"
assert int(v018_protocol["dimension"]) == 384
assert v018_protocol["test_accessed"] is False
assert v018_protocol["test_relevance_accessed"] is False

CONTRASTS = [
    {"label": "ef32_to_256", "low_ef": 32, "high_ef": 256},
    {"label": "ef64_to_256", "low_ef": 64, "high_ef": 256},
]
SENSITIVITY_PROTOCOL = {
    "study_id": "ARC-v0.20d-HNSW-SEVERITY-SENSITIVITY",
    "status": "POST_PRIMARY_EXPLORATORY_CONTRASTS_FIXED_BEFORE_SECONDARY_TRAJECTORY_OUTCOMES",
    "parent_study": "ARC-v0.20c-HNSW-FULL",
    "primary_result_immutable": {
        "low_efSearch": 8, "high_efSearch": 256, "mechanism_gate": "HNSW_REVERSAL",
        "primary_freeze_sha256": sha256_file(PRIMARY_FREEZE),
        "primary_report_sha256": sha256_file(PRIMARY_REPORT),
    },
    "secondary_contrasts": CONTRASTS,
    "selection_rationale": "Milder search-effort contrasts against the same high-fidelity efSearch=256 to audit whether the qualitative reversal sign pattern persists as severity decreases.",
    "feedback_grid": {"alphas":[0.1,0.3,0.5,0.7],"mean_k":[5,20,50],"softmax_k":[5,20],"temperatures":[0.05,0.1,0.2,0.5],"policy_count":44,"rounds":4,"top_k":100,"utility":"nDCG@10","epsilon":0.002},
    "inference": {"sampling_unit":"query","bootstrap_reps":10000,"primary_secondary_endpoints":["H1","H2","H3abs","H3signed"]},
    "source_hashes": {"hnsw_index_sha256":sha256_file(INDEX_PATH),"v018_protocol_sha256":sha256_file(V018_PROTOCOL),"v018_manifest_sha256":sha256_file(V018_MANIFEST),"fit_membership_sha256":v018_protocol["fit_membership_sha256"],"validation_membership_sha256":v018_protocol["validation_membership_sha256"]},
    "test_accessed": False, "test_relevance_accessed": False,
}
PROTOCOL_PATH = OUT / "V020D_SEVERITY_PROTOCOL.json"
if PROTOCOL_PATH.exists():
    old = json.loads(PROTOCOL_PATH.read_text())
    assert old["secondary_contrasts"] == CONTRASTS
else:
    PROTOCOL_PATH.write_text(json.dumps(SENSITIVITY_PROTOCOL, indent=2))
print(json.dumps(SENSITIVITY_PROTOCOL, indent=2))
print("SECONDARY CONTRAST LIST FROZEN")


## 3. Load one-shot ladder and quantify contrast severity

In [ ]:
ladder = pd.read_csv(ONE_SHOT_LADDER).sort_values("efSearch")
display(ladder)
one_shot_map = dict(zip(ladder["efSearch"].astype(int), ladder["mean_ndcg10"].astype(float)))
severity_rows = []
for c in CONTRASTS:
    lo, hi = c["low_ef"], c["high_ef"]
    severity_rows.append({"contrast":c["label"],"low_ef":lo,"high_ef":hi,"low_fit_ndcg10":one_shot_map[lo],"high_fit_ndcg10":one_shot_map[hi],"one_shot_abs_gap":one_shot_map[hi]-one_shot_map[lo]})
severity_rows.insert(0,{"contrast":"PRIMARY_ef8_to_256","low_ef":8,"high_ef":256,"low_fit_ndcg10":one_shot_map[8],"high_fit_ndcg10":one_shot_map[256],"one_shot_abs_gap":one_shot_map[256]-one_shot_map[8]})
severity_df = pd.DataFrame(severity_rows)
severity_df.to_csv(OUT/"v020d_one_shot_severity_ladder.csv", index=False)
display(severity_df)


## 4. Load HNSW index, queries, split, qrels, and document IDs

In [ ]:
def read_id_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]
def canonical_membership_hash(ids):
    return hashlib.sha256("\n".join(sorted(map(str,ids))).encode("utf-8")).hexdigest()

index = faiss.read_index(str(INDEX_PATH))
assert index.ntotal == 5_416_568
doc_ids = np.asarray(read_id_lines(DOC_IDS_PATH), dtype=object)
assert len(doc_ids) == index.ntotal
queries = np.load(V018_QUERY_EMB, mmap_mode="r")
query_ids = np.asarray(read_id_lines(V018_QUERY_IDS), dtype=object)
assert queries.shape == (6666,384)
split_df = pd.read_csv(V013_SPLIT)
split_df["query_id"] = split_df["query_id"].astype(str)
labels = split_df["split"].astype(str).str.lower()
fit_ids = set(split_df.loc[labels=="fit","query_id"])
val_ids = set(split_df.loc[labels=="validation","query_id"])
assert len(fit_ids)==3350 and len(val_ids)==3316
assert canonical_membership_hash(fit_ids)==v018_protocol["fit_membership_sha256"]
assert canonical_membership_hash(val_ids)==v018_protocol["validation_membership_sha256"]

qrels = pd.read_csv(FEVER_QRELS_DEV, sep="\t")
rename={}
for c in qrels.columns:
    z=c.lower().replace("_","-")
    if z in {"query-id","queryid","qid"}: rename[c]="query_id"
    elif z in {"corpus-id","corpusid","doc-id","docid"}: rename[c]="doc_id"
    elif z in {"score","relevance","rel"}: rename[c]="relevance"
qrels=qrels.rename(columns=rename)
if not {"query_id","doc_id","relevance"} <= set(qrels.columns):
    qrels=pd.read_csv(FEVER_QRELS_DEV, sep="\t", names=["query_id","doc_id","relevance"])
qrels=qrels[["query_id","doc_id","relevance"]].copy()
qrels["query_id"]=qrels["query_id"].astype(str); qrels["doc_id"]=qrels["doc_id"].astype(str); qrels["relevance"]=qrels["relevance"].astype(float)
QREL_MAP={str(qid):{str(d):float(r) for d,r in zip(sub["doc_id"],sub["relevance"])} for qid,sub in qrels.groupby("query_id")}
print("HNSW READY:", index.ntotal)
print("FIT:", len(fit_ids), "VAL:", len(val_ids))
print("INPUT + SPLIT AUDIT: PASS")


## 5. Frozen 44-policy grid

In [ ]:
CONFIG={"top_k":100,"utility_k":10,"rounds":4,"epsilon":0.002,"alphas":[0.1,0.3,0.5,0.7],"mean_k":[5,20,50],"softmax_k":[5,20],"temperatures":[0.05,0.1,0.2,0.5],"bootstrap_reps":10000,"checkpoint_every_queries":25,"seed":20260820}
def make_policies():
    out=[]
    for a in CONFIG["alphas"]:
        for k in CONFIG["mean_k"]: out.append({"family":"mean","alpha":a,"k":k,"temperature":np.nan})
        for k in CONFIG["softmax_k"]:
            for tau in CONFIG["temperatures"]: out.append({"family":"softmax","alpha":a,"k":k,"temperature":tau})
    assert len(out)==44
    return out
POLICIES=make_policies()
print("policy count:", len(POLICIES))


## 6. Coupled trajectory implementation

In [ ]:
def normalize_vec(x, eps=1e-12):
    x=np.asarray(x,dtype=np.float32); n=float(np.linalg.norm(x)); return x/max(n,eps)
def hnsw_search(q, ef, k=100):
    index.hnsw.efSearch=int(ef); return index.search(np.asarray(q,dtype=np.float32),int(k))
def dcg(rels):
    rels=np.asarray(rels,dtype=np.float64)
    if len(rels)==0: return 0.0
    return float(np.sum((np.power(2.0,rels)-1.0)*(1.0/np.log2(np.arange(2,len(rels)+2)))))
def ndcg_at_k(retrieved_doc_ids,qrel_dict,k=10):
    rels=[qrel_dict.get(str(d),0.0) for d in retrieved_doc_ids[:k]]; ideal=sorted(qrel_dict.values(),reverse=True)[:k]; denom=dcg(ideal); return 0.0 if denom==0 else dcg(rels)/denom
def jaccard_distance(a,b):
    A,B=set(map(int,a)),set(map(int,b)); return 1.0-len(A&B)/max(1,len(A|B))
def ols_slope(y):
    y=np.asarray(y,dtype=np.float64); x=np.arange(len(y),dtype=np.float64); xm,ym=x.mean(),y.mean(); return float(np.sum((x-xm)*(y-ym))/np.sum((x-xm)**2))
_SHARD_CACHE={}
def get_corpus_rows(rows):
    rows=np.asarray(rows,dtype=np.int64); out=np.empty((len(rows),384),dtype=np.float32); by_shard={}
    for oi,row in enumerate(rows):
        if row<5_400_000: sid,off=int(row//100_000),int(row%100_000)
        else: sid,off=54,int(row-5_400_000)
        by_shard.setdefault(sid,[]).append((oi,off))
    for sid,pairs in by_shard.items():
        if sid not in _SHARD_CACHE: _SHARD_CACHE[sid]=np.load(V018_SHARDS/f"shard-{sid:04d}.float16.npy",mmap_mode="r")
        mm=_SHARD_CACHE[sid]; outs=[a for a,_ in pairs]; offs=[b for _,b in pairs]; out[outs]=np.asarray(mm[offs],dtype=np.float32)
    return out
def feedback_vector(doc_vecs,scores,family,temperature):
    x=np.asarray(doc_vecs,dtype=np.float32)
    if family=="mean": f=x.mean(axis=0)
    else:
        z=np.asarray(scores,dtype=np.float64)/float(temperature); z-=z.max(); w=np.exp(z); w/=w.sum(); f=(x*w[:,None]).sum(axis=0)
    return normalize_vec(f)
def update_state(q0,f,alpha): return normalize_vec((1.0-alpha)*q0+alpha*f)
def run_single_pair(q0,qid,policy,low_ef,high_ef,contrast_label):
    q0=normalize_vec(q0); qL,qH=q0.copy(),q0.copy(); d_hist=[]; j_hist=[]; a_hist=[]; g_hist=[]; uL_hist=[]; uH_hist=[]
    for t in range(CONFIG["rounds"]+1):
        sL,iL=hnsw_search(qL[None,:],low_ef,CONFIG["top_k"]); sH,iH=hnsw_search(qH[None,:],high_ef,CONFIG["top_k"]); sL,iL=sL[0],iL[0]; sH,iH=sH[0],iH[0]
        iLv,sLv=iL[iL>=0],sL[iL>=0]; iHv,sHv=iH[iH>=0],sH[iH>=0]
        docsL=[doc_ids[j] for j in iLv]; docsH=[doc_ids[j] for j in iHv]; qr=QREL_MAP.get(str(qid),{})
        uL=ndcg_at_k(docsL,qr,CONFIG["utility_k"]); uH=ndcg_at_k(docsH,qr,CONFIG["utility_k"])
        d_hist.append(1.0-float(np.dot(qL,qH))); j_hist.append(jaccard_distance(iLv,iHv)); a_hist.append(abs(uH-uL)); g_hist.append(uH-uL); uL_hist.append(uL); uH_hist.append(uH)
        if t==CONFIG["rounds"]: break
        k=int(policy["k"])
        fL=feedback_vector(get_corpus_rows(iLv[:k]),sLv[:k],policy["family"],policy["temperature"]); fH=feedback_vector(get_corpus_rows(iHv[:k]),sHv[:k],policy["family"],policy["temperature"])
        qL=update_state(q0,fL,policy["alpha"]); qH=update_state(q0,fH,policy["alpha"])
    return {"contrast":contrast_label,"low_ef":int(low_ef),"high_ef":int(high_ef),"query_id":str(qid),"family":policy["family"],"alpha":float(policy["alpha"]),"k":int(policy["k"]),"temperature":float(policy["temperature"]) if not pd.isna(policy["temperature"]) else np.nan,"H1":ols_slope(d_hist),"H2":ols_slope(j_hist),"H3abs":ols_slope(a_hist),"H3signed":ols_slope(g_hist),"final_uL":float(uL_hist[-1]),"final_uH":float(uH_hist[-1]),"final_signed_gap":float(uH_hist[-1]-uL_hist[-1])}


## 7. Resumable runner for both contrasts and both splits

In [ ]:
def query_mask(ids,selected):
    selected=set(map(str,selected)); return np.asarray([str(x) in selected for x in ids],dtype=bool)
def run_contrast_split_resumable(contrast,split_name,selected_ids):
    label=contrast["label"]; low_ef=contrast["low_ef"]; high_ef=contrast["high_ef"]; split_dir=OUT/label/split_name; split_dir.mkdir(parents=True,exist_ok=True)
    mask=query_mask(query_ids,selected_ids); q=np.asarray(queries[mask],dtype=np.float32); qids=query_ids[mask]; chunk=CONFIG["checkpoint_every_queries"]
    for start in range(0,len(q),chunk):
        stop=min(start+chunk,len(q)); cp=split_dir/f"{split_name}_{start:04d}_{stop:04d}.parquet"
        if cp.exists(): print("skip",label,cp.name); continue
        rows=[]; t0=time.perf_counter()
        for qi in range(start,stop):
            for policy in POLICIES: rows.append(run_single_pair(q[qi],qids[qi],policy,low_ef,high_ef,label))
        tmp=cp.with_suffix(".tmp.parquet"); pd.DataFrame(rows).to_parquet(tmp,index=False); os.replace(tmp,cp); print(label,cp.name,f"{time.perf_counter()-t0:.1f}s")
    parts=sorted(split_dir.glob(f"{split_name}_*.parquet")); df=pd.concat([pd.read_parquet(p) for p in parts],ignore_index=True); assert len(df)==len(q)*44; assert df["query_id"].nunique()==len(q)
    merged=OUT/f"v020d_{label}_{split_name}_endpoints.parquet"; df.to_parquet(merged,index=False); return df
RESULTS={}
for contrast in CONTRASTS:
    label=contrast["label"]; RESULTS[(label,"FIT")]=run_contrast_split_resumable(contrast,"fit",fit_ids); RESULTS[(label,"validation")]=run_contrast_split_resumable(contrast,"validation",val_ids)
print("ALL SECONDARY TRAJECTORIES COMPLETE")


## 8. Query-cluster bootstrap endpoint summary

In [ ]:
def bootstrap_endpoint(df,metric,reps=10000,seed=20260820):
    vals=df.groupby("query_id",sort=False)[metric].mean().to_numpy(dtype=np.float64); obs=float(vals.mean()); rng=np.random.default_rng(seed); n=len(vals); boot=np.empty(reps,dtype=np.float64)
    for b in range(reps):
        idx=rng.integers(0,n,size=n); boot[b]=vals[idx].mean()
    lo,hi=np.quantile(boot,[0.025,0.975]); return obs,float(lo),float(hi)
summary_rows=[]
primary_ep=pd.read_csv(V020C/"v020c_aggregate_endpoints.csv")
for _,r in primary_ep[primary_ep["split"]=="validation"].iterrows():
    summary_rows.append({"contrast":"PRIMARY_ef8_to_256","split":"validation","metric":r["metric"],"mean":r["mean"],"ci_low":r["ci_low"],"ci_high":r["ci_high"],"source":"frozen v0.20c"})
for (label,split_name),df in RESULTS.items():
    for i,metric in enumerate(["H1","H2","H3abs","H3signed"]):
        mean,lo,hi=bootstrap_endpoint(df,metric,reps=CONFIG["bootstrap_reps"],seed=CONFIG["seed"]+i)
        summary_rows.append({"contrast":label,"split":split_name,"metric":metric,"mean":mean,"ci_low":lo,"ci_high":hi,"source":"v0.20d exploratory sensitivity"})
summary_df=pd.DataFrame(summary_rows); summary_df.to_csv(OUT/"v020d_endpoint_summary.csv",index=False); display(summary_df)


## 9. Qualitative sign-pattern audit

In [ ]:
def sign_gate(row):
    if row["ci_low"]>0: return "positive"
    if row["ci_high"]<0: return "negative"
    return "contains_zero"
validation_summary=summary_df[(summary_df["split"]=="validation") & (summary_df["contrast"].isin([c["label"] for c in CONTRASTS]))].copy(); validation_summary["ci_sign"]=validation_summary.apply(sign_gate,axis=1)
pivot=validation_summary.pivot(index="contrast",columns="metric",values="ci_sign").reset_index()
def pattern_label(r):
    h1,h2,h3=r.get("H1"),r.get("H2"),r.get("H3abs")
    if h1=="positive" and h2=="negative" and h3=="negative": return "PERSISTS_H1_POS_H2_NEG_H3ABS_NEG"
    if h3=="negative": return "H3ABS_CONTRACTION_ONLY"
    if h3=="contains_zero": return "H3ABS_NULL"
    return "H3ABS_POSITIVE"
pivot["qualitative_result"]=pivot.apply(pattern_label,axis=1); pivot.to_csv(OUT/"v020d_qualitative_sign_audit.csv",index=False); display(pivot)


## 10. Regime prevalence at epsilon=0.002

In [ ]:
def classify_regime(x,eps=0.002):
    if x>eps: return "amplifying"
    if x<-eps: return "contracting"
    return "stable_null"
regime_rows=[]
for (label,split_name),df in RESULTS.items():
    reg=df["H3abs"].map(classify_regime); counts=reg.value_counts(); n=len(reg)
    regime_rows.append({"contrast":label,"split":split_name,"stable_null_pct":100*counts.get("stable_null",0)/n,"amplifying_pct":100*counts.get("amplifying",0)/n,"contracting_pct":100*counts.get("contracting",0)/n,"n_events":n})
regime_df=pd.DataFrame(regime_rows); regime_df.to_csv(OUT/"v020d_regime_summary.csv",index=False); display(regime_df)


## 11. Final severity report + artifact hashes

In [ ]:
final_report={"study_id":"ARC-v0.20d-HNSW-SEVERITY-SENSITIVITY","status":"COMPLETE_POST_PRIMARY_EXPLORATORY_SENSITIVITY","primary_hnsw_result_unchanged":True,"primary_parent":{"contrast":"ef8_to_256","gate":primary_report["mechanism_gate"],"freeze_sha256":sha256_file(PRIMARY_FREEZE),"report_sha256":sha256_file(PRIMARY_REPORT)},"secondary_contrasts":CONTRASTS,"one_shot_severity":severity_df.to_dict(orient="records"),"endpoint_summary":summary_df.to_dict(orient="records"),"qualitative_sign_audit":pivot.to_dict(orient="records"),"regime_summary":regime_df.to_dict(orient="records"),"interpretation_constraint":"These are post-primary exploratory severity sensitivity results. They test robustness of the qualitative HNSW reversal pattern but do not replace or upgrade the frozen v0.20c primary claim gate.","test_accessed":False,"test_relevance_accessed":False,"software":{"faiss":faiss.__version__,"numpy":np.__version__,"pandas":pd.__version__,"python":sys.version.split()[0]}}
REPORT_PATH=OUT/"v020d_final_report.json"; REPORT_PATH.write_text(json.dumps(final_report,indent=2))
artifact_paths=[PROTOCOL_PATH,OUT/"v020d_one_shot_severity_ladder.csv",OUT/"v020d_endpoint_summary.csv",OUT/"v020d_qualitative_sign_audit.csv",OUT/"v020d_regime_summary.csv",REPORT_PATH]
for c in CONTRASTS:
    for split_name in ["fit","validation"]: artifact_paths.append(OUT/f"v020d_{c['label']}_{split_name}_endpoints.parquet")
hash_rows=[]
for p in artifact_paths:
    if p.exists(): hash_rows.append({"file":p.name,"bytes":p.stat().st_size,"sha256":sha256_file(p)})
hash_df=pd.DataFrame(hash_rows); hash_df.to_csv(OUT/"V020D_ARTIFACT_SHA256.csv",index=False)
print(json.dumps(final_report,indent=2)[:8000]); display(hash_df); print("FINAL REPORT:",REPORT_PATH)


# Paper-facing decision rule

Use this audit only to answer the concern that the primary `8 -> 256` HNSW contrast may be unusually severe.

A defensible write-up is:

> “The frozen primary HNSW contrast was intentionally left unchanged. In a post-primary severity audit, we additionally evaluated milder `efSearch` contrasts against the same high-fidelity comparator. These analyses are exploratory and do not modify the primary gate.”

If both milder contrasts retain `H1 > 0`, `H2 < 0`, and `H3abs < 0` with query-cluster CIs excluding zero, the paper may state that the qualitative reversal persists across the tested HNSW search-effort severities.

If one or both become null or change sign, report that boundary honestly rather than averaging across contrasts.
